# Week 3, day 2 — Extra practice 05 SOLUTIONS: pivot_table   (L04)

Executed in the lab image (pandas 3.0.5) against the real
`../data/orders_long.csv`. Every quoted number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 05 — pivot_table. Run this once.
import pandas as pd

orders = pd.read_csv("../data/orders_long.csv")

print("rows:", len(orders))
print("categories:", list(orders["Category"].unique()))
print("ship modes:", list(orders["ShipMode"].unique()))

### Question 1

A `(3, 3)` table; grand total `1605576.22`, matching the raw total.

The reconciliation check from worksheet 05 Q5. It works because the
aggregate is a sum and every row falls into exactly one cell.

In [ ]:
pt = orders.pivot_table(index="Category", columns="ShipMode",
                        values="Sales", aggfunc="sum")
print(pt.round(2).to_string())
print()
print("shape:", pt.shape)
print("grand total:", round(pt.sum().sum(), 2))
print("raw total:  ", round(orders["Sales"].sum(), 2))

### Question 2

Technology / Regular Air -> `sum 538247.63`, `mean 2078.18`, `count 259.00`, `max 10281.79`.

One cell, four numbers, four different questions: how much did we sell,
what is a typical order, how many transactions, what was the biggest.

The count is the one that makes the others readable. `2078.18` means
something quite different over 259 orders than it would over 3, and the
mean alone does not say which.

In [ ]:
for fn in ("sum", "mean", "count", "max"):
    pt = orders.pivot_table(index="Category", columns="ShipMode",
                            values="Sales", aggfunc=fn)
    print("%-6s -> %12.2f" % (fn, pt.loc["Technology", "Regular Air"]))
print()
sub = orders[(orders["Category"] == "Technology")
             & (orders["ShipMode"] == "Regular Air")]["Sales"]
print("rows behind that cell:", len(sub))

### Question 3

`values=["Sales", "Profit"]` -> `(3, 6)` with a **2-level** column index like `('Profit', 'Delivery Truck')`.

Two value columns x three ship modes = six columns, addressed by tuples.

Note the order: `Profit` comes before `Sales` because the levels are sorted
alphabetically, not in the order you listed them. If you are reading the
table by position, that will catch you.

In [ ]:
pt = orders.pivot_table(index="Category", columns="ShipMode",
                        values=["Sales", "Profit"], aggfunc="sum")
print("shape:", pt.shape, "| column levels:", pt.columns.nlevels)
print("first 3 columns:", list(pt.columns[:3]))
print()
print(pt.loc["Furniture"].round(2).to_string())

### Question 4

`aggfunc={"Sales": "sum", "Profit": "mean"}` -> totals for sales, averages for profit, in one table.

This is usually the right shape for a report: money in as a total, margin
as an average. Passing a single `aggfunc` for both would force one of them
to be wrong.

A dict of aggregates is worth reaching for whenever the columns mean
different kinds of thing.

In [ ]:
pt = orders.pivot_table(index="Category", columns="ShipMode",
                        values=["Sales", "Profit"],
                        aggfunc={"Sales": "sum", "Profit": "mean"})
print(pt.round(2).to_string())

### Question 5

The `All` row equals the sum of the cells above it — `874` both ways. Grand total `1093`.

Unlike worksheet 05 Q8, the margin agrees with the naive check here — and
for a real reason. `margins` computes from the underlying rows, and counting
rows is *additive*: the count of a union of disjoint groups is the sum of
their counts.

Means are not additive, which is why the mean margin disagreed. Sums and
counts are safe; anything involving a division is not.

In [ ]:
pt = orders.pivot_table(index="Category", columns="ShipMode",
                        values="Sales", aggfunc="count", margins=True)
print(pt.to_string())
print()
col = pt["Regular Air"].drop("All")
print("sum of the cells:", int(col.sum()))
print("the 'All' margin: ", int(pt.loc["All", "Regular Air"]))
print("equal:", col.sum() == pt.loc["All", "Regular Air"])

### Question 6

Margin percentages from `4.2` to `22.3`. -> **`0`** negative cells.

Dividing one pivot by another works because both have the same index and
columns, so they align cell for cell.

Every category-shipmode combination is profitable in aggregate. Hold that
next to Q7.

In [ ]:
profit = orders.pivot_table(index="Category", columns="ShipMode",
                            values="Profit", aggfunc="sum")
sales = orders.pivot_table(index="Category", columns="ShipMode",
                           values="Sales", aggfunc="sum")
margin = (profit / sales * 100).round(1)
print(margin.to_string())
print()
print("negative cells:", int((margin < 0).sum().sum()), "of", margin.size)

### Question 7

Furniture / Delivery Truck -> ratio of sums **`4.2`**, mean of ratios **`-12.8`**. -> Regular Air `11.7` vs `3.1`.

Two numbers with opposite signs describing the same cell.

**Ratio of sums** — total profit divided by total sales — is the margin.
It weights every order by its size, which is what a margin means.

**Mean of ratios** averages each order's own margin percentage, weighting a
five-pound order equally with a five-thousand-pound one. A handful of tiny
loss-making orders with large negative percentages drags it below zero while
the business is comfortably profitable.

Both are one line of Pandas and both look like 'the margin'. The mean of
ratios is almost never the number you want, and it is the easier one to
write by accident — just add a per-row column and average it.

In [ ]:
work = orders.copy()
work["RowMargin"] = work["Profit"] / work["Sales"] * 100

ratio_of_sums = (orders.pivot_table(index="Category", columns="ShipMode",
                                    values="Profit", aggfunc="sum")
                 / orders.pivot_table(index="Category", columns="ShipMode",
                                      values="Sales", aggfunc="sum") * 100)
mean_of_ratios = work.pivot_table(index="Category", columns="ShipMode",
                                  values="RowMargin", aggfunc="mean")

print("ratio of sums  (Furniture):")
print(ratio_of_sums.loc["Furniture"].round(1).to_string())
print()
print("mean of ratios (Furniture):")
print(mean_of_ratios.loc["Furniture"].round(1).to_string())

### Question 8

`aggfunc="total"` -> **raises** `AttributeError: 'total' is not a valid function for 'DataFrameGroupBy'`.

There is no `total`; the name is `sum`. The message is slightly indirect —
it mentions `DataFrameGroupBy` rather than `pivot_table` — because
`pivot_table` is implemented on top of `groupby` and the string is passed
straight through.

Worth recognising, because it means anything valid in `groupby.agg()` is
valid here: built-in names, NumPy functions, or your own callable.

In [ ]:
print(orders.pivot_table(index="Category", columns="ShipMode",
                         values="Sales", aggfunc="total"))